# CS236781: Deep Learning on Computational Accelerators
# Final Project

Faculty of Computer Science, Technion.

Submitted by:

| #       |              Name |             Id |             email                  |
|---------|-------------------|----------------|----------------------------------- |
|Student 1|  Daniel Elgarici  |   305341828    | elgarici-dan@campus.technion.ac.il |
|Student 2|  Tal Benjo        |   318655701    | tal.benjo@campus.technion.ac.il    |

## Introduction

In this project we strive to reproduce and rexamine key results from the article:

[No Data, No Optimization: A Lightweight Method to Disrupt Neural Networks with Sign-Flips (Galil et al., 2025)](https://arxiv.org/abs/2502.07408)

The article focused its efforts into visual models and CNNs in particular, in our work we'll extend tihs into the realm of LLMs focusing on BERT, as a proof of concept for the validity and feasibility of such attacks on language models.

# Reproducing a key result from the article
Our first order of business is to implement the attack strategies discussed in the article, see if we can reproduce the findings in the paper, and by doing so validate both our implementation and the results. One of the most staggering findings in the article is that even with a minimal change of sign bits you can substantially damage the model and alter its output. A concrete result discussed in the article was targeting ShuffleNetV2 with DNL attack. 

We reran the very same experiment to test our implementation and verify the results.

In [1]:
# If needed:
# !pip install torch torchvision --quiet

import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

VAL_DIR   = "/home/elgarici-dan/project_deep/dataset/val_final"  # organized as val/<synset>/*.JPEG
BATCH_SIZE = 256
NUM_WORKERS = 8
RESIZE, CROP = 256, 224

tfm = transforms.Compose([
    transforms.Resize(RESIZE),
    transforms.CenterCrop(CROP),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
]) # The normalization values are based on the statistics of the training data
# We rescale, crop and normalize to ensure the model seens inputs in the same distribution it was trained on

val_ds = datasets.ImageFolder(VAL_DIR, transform=tfm)

In [2]:
@torch.no_grad()
def calc_acc(model, loader, device=DEVICE):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        pred = model(x).argmax(1)
        correct += (pred == y).sum().item()
        total   += y.numel()
    return 100. * correct / max(1, total)

# Extracts first L (learnable!!!) layers for CNN
def first_L_param_layers_CNN(model, L):
    layers = []
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            layers.append(m)
    return layers[:L]
    
def collect_dnl_passfree_candidates_CNN(module):
    W = module.weight.data
    A = W.abs()
    cands = []
    if isinstance(module, nn.Conv2d):
        OC, IC, KH, KW = W.shape
        flat = A.view(OC, IC, -1)
        vmax, arg = flat.max(dim=2)  # (OC, IC)
        for oc in range(OC):
            for ic in range(IC):
                s = float(vmax[oc, ic])
                if s == 0.0: continue
                p = int(arg[oc, ic])
                kh, kw = divmod(p, KW)
                cands.append((s, (oc, ic, kh, kw), module))
    elif isinstance(module, nn.Linear):
        vmax, arg = A.max(dim=1)     # per output row
        for o in range(W.size(0)):
            s = float(vmax[o])
            if s == 0.0: continue
            j = int(arg[o])
            cands.append((s, (o, j), module))
    return cands

@torch.no_grad()
def flip_sign_(module, index):
    w = module.weight.data
    w[index] = -w[index]


def eval_dnl_ARk(model, loader, L=10, ks=(1,2,3,4,5), device='cuda',
                 collector='passfree'):
    """
    collector: 'passfree' -> use collect_dnl_passfree_candidates_CNN (|w|)
               '1pass'    -> use collect_dnl_passfree_candidates_CNN (needs grads prepared)
    """
    model = model.to(device).eval()
    clean = calc_acc(model, loader, device)

    # choose collector
    if collector == 'passfree':
        get_cands = collect_dnl_passfree_candidates_CNN
    elif collector == '1pass':
        get_cands = collect_dnl_passfree_candidates_CNN
    else:
        raise ValueError("collector must be 'passfree' or '1pass'.")

    # build candidate list from first L param layers
    cands = []
    for m in first_L_param_layers_CNN(model, L):
        cands.extend(get_cands(m))
    cands.sort(key=lambda t: t[0], reverse=True)  # global top by score

    acc_k = {}
    flipped = 0
    for k in ks:
        while flipped < k:
            _, idx, mod = cands[flipped]
            flip_sign_(mod, idx)
            flipped += 1
        acc_k[k] = calc_acc(model, loader, device)

    ar_k = {k: (clean - acc_k[k]) / max(1e-12, clean) for k in ks}
    return clean, acc_k, ar_k, len(cands)

In [31]:
weights = models.ShuffleNet_V2_X1_0_Weights.DEFAULT
model = models.shufflenet_v2_x1_0(weights=weights)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

L = 10
KS = (1,2,3,4,5)
clean, acc_k, ar_k, num_cands = eval_dnl_ARk(model, val_loader, L=L, ks=KS, device=DEVICE)

print("Evaluating pass-free attack on ShuffleNet:")
print(f"Evaluated on {len(val_ds)} images.")
print(f"Baseline top-1 accuracy: {clean:.3f}%")
print(f"Candidates in first {L} layers: {num_cands}")
print("\nAR(k) (cumulative flips):")
print("k\tAcc_k (%)\tAR(k)")
for k in KS:
    print(f"{k}\t{acc_k[k]:.3f}\t\t{ar_k[k]*100:.2f}%")


/home/elgarici-dan/miniconda3/envs/cs236781-hw/lib/python3.8/site-packages/torch/utils/data/dataloader.py:557: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


Evaluating pass-free attack on ShuffleNet:
Evaluated on 50000 images.
Baseline top-1 accuracy: 69.356%
Candidates in first 10 layers: 16452

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	4.210		93.93%
2	0.288		99.58%
3	0.264		99.62%
4	0.216		99.69%
5	0.168		99.76%


In [3]:
def dnl_1pass_prepare_grads(model, device='cuda', batch_size=32, H=224, W=224):
    """
    Runs a single forward+backward pass on Gaussian input to populate .grad for all params.
    Uses R(θ) = sum of logits over the batch (Algorithm 2).
    """
    model.to(device).train()  # train/eval doesn't matter for grads; keep BN affine behavior consistent
    for p in model.parameters():
        if p.grad is not None:
            p.grad.zero_()

    X = torch.randn(batch_size, 3, H, W, device=device)  # Gaussian input
    logits = model(X)
    R = logits.sum()  # sum of logits across batch and classes
    R.backward()      # populates .grad for parameters
    model.eval()      # return to eval for accuracy measurements
    
    
def collect_dnl1pass_candidates(module):
    """
    One candidate per 'kernel', scored by the 1P-DNL hybrid score:
      S = |w| + |(w * g) + 0.5 * w^2 * g^2|
    where g = dR/dw, with R = sum of logits on Gaussian input.
    Returns list of tuples (score, index_tuple, module), same as your DNL collector.
    """
    W = module.weight.data
    G = module.weight.grad
    if G is None:
        # user forgot to call dnl_1pass_prepare_grads(model, ...)
        raise RuntimeError("No gradients found on module.weight. Call dnl_1pass_prepare_grads(model, ...) first.")

    # S = |w| + |(w*g) + 0.5*w^2*g^2|  (Gauss–Newton-like diagonal Hessian approx)
    S = W.abs() + ((W * G) + 0.5 * (W * W) * (G * G)).abs()

    cands = []
    if isinstance(module, nn.Conv2d):
        OC, IC, KH, KW = W.shape
        flat = S.view(OC, IC, -1)
        vmax, arg = flat.max(dim=2)  # pick the best element per (out_ch, in_ch) kernel
        for oc in range(OC):
            for ic in range(IC):
                s = float(vmax[oc, ic])
                if s == 0.0: 
                    continue
                p = int(arg[oc, ic])
                kh, kw = divmod(p, KW)
                cands.append((s, (oc, ic, kh, kw), module))
    elif isinstance(module, nn.Linear):
        vmax, arg = S.max(dim=1)     # best column per output row
        OUT = W.size(0)
        for o in range(OUT):
            s = float(vmax[o])
            if s == 0.0:
                continue
            j = int(arg[o])
            cands.append((s, (o, j), module))
    return cands

In [32]:
weights = models.ShuffleNet_V2_X1_0_Weights.DEFAULT
model = models.shufflenet_v2_x1_0(weights=weights)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

L = 10
KS = (1,2,3,4,5)
dnl_1pass_prepare_grads(model, device=DEVICE, batch_size=32, H=224, W=224)
_, acc_k, ar_k, num_cands = eval_dnl_ARk(model, val_loader, L=L, ks=KS, device=DEVICE, collector="1pass")

print("Evaluating 1pass attack on ShuffleNet:")
print(f"Evaluated on {len(val_ds)} images.")
print(f"Baseline top-1 accuracy: {clean:.3f}%")
print(f"Candidates in first {L} layers: {num_cands}")
print("\nAR(k) (cumulative flips):")
print("k\tAcc_k (%)\tAR(k)")
for k in KS:
    print(f"{k}\t{acc_k[k]:.3f}\t\t{ar_k[k]*100:.2f}%")

Evaluating 1pass attack on ShuffleNet:
Evaluated on 50000 images.
Baseline top-1 accuracy: 63.782%
Candidates in first 10 layers: 16452

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	4.306		93.25%
2	0.350		99.45%
3	0.256		99.60%
4	0.180		99.72%
5	0.188		99.71%


# Conclusion
As we observe in the results above we managed to recreate the same damage with minimal change to the model. Now, that we have validated our own implementation we can move forward and extend the research into LLMs and see how the same type of attacks can damage them.


In [4]:
# !pip install torch torchvision transformers datasets
import os, math, random, torch, torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, logging
from typing import List, Tuple, Dict

logging.set_verbosity_error()   # hide all warnings/errors except fatal

os.environ["TOKENIZERS_PARALLELISM"] = "false"
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
torch.manual_seed(SEED); random.seed(SEED)

## Testing different strategies for choosing weights
### Change only 1 type of weight: from Q,K,V
### Interlacing (?)
### Targeting only the MLP headof each transformer

#### todo:Running on more models and creating boxplot

In [5]:
def load_sst2_llm(model_name: str = "textattack/bert-base-uncased-SST-2",
                   batch_size: int = 64,
                   max_train: int = None,
                   max_eval: int = None):
    """
    Loads SST-2 (GLUE) and returns: tokenizer, eval_loader
    (We evaluate on the validation/dev split for AR(k).)
    """
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    ds = load_dataset("glue", "sst2")

    # n_params = sum(p.numel() for p in model.parameters())
    # print(f"{n_params:,} parameters")  
    
    def tokenize(batch):
        return tok(batch["sentence"], truncation=True, padding=False, max_length=128)

    # tokenize; keep 'label' by removing only the text column
    ds_tok = ds.map(tokenize, batched=True, remove_columns=["sentence"])

    # some tokenizers return token_type_ids (BERT) and some don't (RoBERTa). It's fine either way.

    # rename 'label' -> 'labels' (what HF models expect for supervised heads)
    if "label" in ds_tok["train"].column_names:
        ds_tok = ds_tok.rename_column("label", "labels")

    # optional subsampling for speed
    def maybe_select(d, n):
        return d.select(range(min(n, len(d)))) if (n is not None) else d

    eval_ds = maybe_select(ds_tok["validation"], max_eval)

    def collate_fn(features):
        # dynamic padding for inputs; labels pass through untouched
        batch = tok.pad(
            {k: [f[k] for f in features] for k in features[0] if k != "labels"},
            return_tensors="pt"
        )
        if "labels" in features[0]:
            batch["labels"] = torch.tensor([f["labels"] for f in features], dtype=torch.long)
        return batch

    eval_loader = DataLoader(
        eval_ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=2,
        pin_memory=True
    )
    return tok, eval_loader

def load_bert_sst2_llm(model_name:str="textattack/bert-base-uncased-SST-2"):
    """
    Loads a BERT already fine-tuned on SST-2.
    """
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    model.to(DEVICE)
    model.eval()
    return model

tokenizer_llm, eval_loader_llm = load_sst2_llm(max_eval=None)  # set a number (e.g., 2000) for faster tests


In [6]:
@torch.no_grad()
def accuracy_llm(model, loader, device=DEVICE) -> float:
    model.eval()
    correct = total = 0
    for batch in loader:
        batch = {k: v.to(device, non_blocking=True) for k,v in batch.items()}
        logits = model(**{k:batch[k] for k in ["input_ids","attention_mask"]}).logits
        pred = logits.argmax(-1)
        correct += (pred == batch["labels"]).sum().item()
        total   += batch["labels"].numel()
    return 100.0 * correct / max(1,total)


In [7]:
def compatible_with_strategy_conditions(layer_name, stratagy) -> bool:
    if stratagy is None:
        return True
    if "Q" in stratagy and "query" in layer_name:
        return True
    if "K" in stratagy and "key" in layer_name:
        return True
    if "V" in stratagy and "value" in layer_name:
        return True
    if (".intermediate.dense" in stratagy or ".output.dense" in stratagy) and "FFN" in layer_name:
        return True
    if "classifier" in stratagy and "classifier" in layer_name:
        return True
    return False    

In [8]:
def first_L_param_layers_llm(model, L:int, stratagy:str or None) -> List[nn.Module]:
    """
    Returns the first L parameter-bearing layers of supported types (nn.Linear, nn.Embedding).
    """
    layers = []
    counter = 0 
    for name, m in model.named_modules():
        if counter >= L:
            break
        # print(name) 
        if isinstance(m, nn.Linear) and compatible_with_strategy_conditions(name, stratagy):
            # must have a weight tensor
            if getattr(m, "weight", None) is not None:
                layers.append(m)
                counter += 1
    print("---LOGGING LAYERS----")
    print(f"Layers actually collected: {len(layers)}")
    print("---END LOGGING LAYERS----")

    return layers
    
def eval_dnl_ARk_llm(model,
                     loader,
                     L:int=10,
                     ks:Tuple[int,...]=(1,2,3,4,5),
                     device=DEVICE,
                     collector:str='passfree',
                     stratagy: str or None = None):
    """
    collector is either 'passfree','1pass'
    stratagy is from {K, Q, V, KQ, KV, QV, KQV, FFN, classifier}
    """
    model.to(device).eval()
    clean = accuracy_llm(model, loader, device)

    # pick collector
    if collector == 'passfree':
        get_cands = collect_passfree_candidates_llm
    elif collector == '1pass':
        get_cands = collect_1pass_candidates_llm
    else:
        raise ValueError("collector must be 'passfree' or '1pass'.")

    # gather candidates from first L modules
    cands = []
    for m in first_L_param_layers_llm(model, L, stratagy):
        cands.extend(get_cands(m))

    cands.sort(key=lambda t: t[0], reverse=True)
    acc_k = {}
    flipped = 0
    for k in ks:
        while flipped < k:
            if flipped >= len(cands):
                break
            _, idx, module = cands[flipped]
            W = module.weight.data
            # print("---LOGGING W[idx] BEFORE FLIPPING----")
            # print(W[idx])
            flip_sign_llm(module, idx)
            flipped += 1
            # print("---LOGGING W[idx] AFTER FLIPPING----")
            # print(W[idx])
        acc_k[k] = accuracy_llm(model, loader, device)

    ar_k = {k: (clean - acc_k[k]) / max(1e-12, clean) for k in ks}
    return clean, acc_k, ar_k, len(cands)


In [9]:
@torch.no_grad()
def flip_sign_llm(module: nn.Module, index: Tuple[int,int]):
    W = module.weight.data
    W[index] = -W[index]

def dnl_1pass_prepare_grads_llm(model, tokenizer, device=DEVICE, batch_size:int=16, seq_len:int=128):
    """
    Single forward+backward pass on random token sequences to populate parameter grads.
    Uses R(θ) = sum of logits over the batch (like the CNN variant uses sum of logits).
    """
    model.to(device).train()
    for p in model.parameters():
        if p.grad is not None:
            p.grad.zero_()

    # Build random token IDs from tokenizer vocab (avoid special token 0 if it’s [PAD])
    vocab_size = tokenizer.vocab_size
    low_id = 5  # skip very low special tokens
    X = torch.randint(low=low_id, high=vocab_size, size=(batch_size, seq_len), device=device)
    attn = torch.ones_like(X, device=device)

    logits = model(input_ids=X, attention_mask=attn).logits
    R = logits.sum()
    R.backward()
    model.eval()


def collect_passfree_candidates_llm(module: nn.Module) -> List[Tuple[float, Tuple[int, ...], nn.Module]]:
    """
    Collect every scalar parameter in the module's weight tensor.
    Returns a list of (score, index_tuple, module)
      - score: absolute value of weight
      - index_tuple: indices in weight tensor (row, col, ...)
      - module: the layer itself
    """
    cands = []
    if isinstance(module, nn.Linear):
        W = module.weight.data
        A = W.abs()

        # flatten and iterate with unravelled indices
        for flat_idx, val in enumerate(A.view(-1)):
            if val == 0:
                continue
            idx = tuple(torch.unravel_index(torch.tensor(flat_idx), A.shape))
            cands.append((float(val), idx, module))
    return cands

def collect_1pass_candidates_llm(module: nn.Module) -> List[Tuple[float, Tuple[int, int], nn.Module]]:
    """
    One-pass (gradient-informed) candidate scoring:
      S = |w| + (w*g) + 0.5 * w^2 * g^2
    with 'one per kernel' = one element per row.
    """
    cands = []
    if isinstance(module, nn.Linear):
        W = module.weight.data
        G = getattr(module.weight, "grad", None)
        if G is None:
            raise RuntimeError("collect_1pass_candidates_llm: missing grads; call dnl_1pass_prepare_grads_llm first.")
        S = W.abs() + ((W * G) + 0.5 * (W * W) * (G * G)).abs()
        # flatten and iterate with unravelled indices
        for flat_idx, val in enumerate(S.view(-1)):
            if val == 0:
                continue
            idx = tuple(torch.unravel_index(torch.tensor(flat_idx), S.shape))
            cands.append((float(val), idx, module))
    return cands


# Expirment #1 - pass free attack, same as the articale.
changed 3 logical layer.
we tryed difrent values of K.


In [9]:
KS = (1,10,100,1_000,10_000, 100_000)
for L in (6, 12, 18): 
    print(f"--- Run Expiriment #1 for L = {L} ---")
    
    model_llm = load_bert_sst2_llm()
    
    # (Optional) quick debug: restrict eval set size earlier with max_eval=2000 in load_sst2_llm
    clean_pf, acck_pf, ark_pf, nC_pf = eval_dnl_ARk_llm(model_llm, eval_loader_llm, L=L, ks=KS, collector='passfree')
    print(ark_pf)
    
    print(f"Evaluating passfree attack on BERT (SST-2):")
    print(f"Evaluated on {len(eval_loader_llm.dataset)} examples.")
    print(f"Baseline top-1 accuracy: {clean_pf:.3f}%")
    print(f"Candidates in first {L} layers: {nC_pf}")
    print("\nAR(k) (cumulative flips):")
    print("k\tAcc_k (%)\tAR(k)")
    
    for k in KS:
        print(f"{k}\t{acck_pf[k]:.3f}\t\t{ark_pf[k]*100:.2f}%")


--- Run Expiriment #1 for L = 6 ---
---LOGGING LAYERS----
Layers actually collected: 6
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0012406947890819732, 1000: 0.0012406947890819732, 10000: 0.0012406947890819732, 100000: 0.0024813895781637925}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 6 layers: 7077888

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.317		0.12%
1000	92.317		0.12%
10000	92.317		0.12%
100000	92.202		0.25%
--- Run Expiriment #1 for L = 12 ---
---LOGGING LAYERS----
Layers actually collected: 12
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0012406947890819732, 1000: 0.003722084367245766, 10000: 0.008684863523573198, 100000: 0.06079404466501254}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 12 layers: 14155776

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
1

# Expirinent #1 results:


# Expirment #2 - 1 pass attack, same as the articale.
changed 3 logical layer.
we tryed difrent values of K.


In [9]:
KS = (1,10,100,1_000,10_000, 100_000)
for L in (6, 12, 18): 
    print(f"--- Run Expiriment #2 for L = {L} ---")
    
    model_llm = load_bert_sst2_llm()

    dnl_1pass_prepare_grads_llm(model_llm, tokenizer_llm, device=DEVICE, batch_size=16, seq_len=128)
    # (Optional) quick debug: restrict eval set size earlier with max_eval=2000 in load_sst2_llm
    clean_pf, acck_pf, ark_pf, nC_pf = eval_dnl_ARk_llm(model_llm, eval_loader_llm, L=L, ks=KS, collector='1pass')
    
    print(f"Evaluating 1pass attack on BERT (SST-2):")
    print(f"Evaluated on {len(eval_loader_llm.dataset)} examples.")
    print(f"Baseline top-1 accuracy: {clean_pf:.3f}%")
    print(f"Candidates in first {L} layers: {nC_pf}")
    print("\nAR(k) (cumulative flips):")
    print("k\tAcc_k (%)\tAR(k)")
    for k in KS:
        print(f"{k}\t{acck_pf[k]:.3f}\t\t{ark_pf[k]*100:.2f}%")

--- Run Expiriment #2 for L = 6 ---
---LOGGING LAYERS----
Layers actually collected: 6
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 6 layers: 7077888

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.317		0.12%
1000	92.202		0.25%
10000	92.661		-0.25%
100000	91.628		0.87%
--- Run Expiriment #2 for L = 12 ---
---LOGGING LAYERS----
Layers actually collected: 12
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 12 layers: 14155776

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.317		0.12%
1000	92.546		-0.12%
10000	92.087		0.37%
100000	82.110		11.17%
--- Run Expiriment #2 for L = 18 ---
---LOGGING LAYERS----
Layers actually collected: 18
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 exam

# Expirinent #2 results:


# Expanding to other attack strategies


In [10]:
def run_experiment(KS_list, LS_list, exp_num, exp_collector, exp_stratagy):
    for L in LS_list: 
        print(f"--- Run Expiriment #{exp_num} for L = {L} ---")
        
        model_llm = load_bert_sst2_llm()
        
        # (Optional) quick debug: restrict eval set size earlier with max_eval=2000 in load_sst2_llm
        clean_pf, acck_pf, ark_pf, nC_pf = eval_dnl_ARk_llm(model_llm, eval_loader_llm, L=L, ks=KS, collector=exp_collector, stratagy = exp_stratagy)
        print(ark_pf)
        
        print(f"Evaluating {exp_collector} attack on BERT (SST-2):")
        print(f"Evaluated on {len(eval_loader_llm.dataset)} examples.")
        print(f"Baseline top-1 accuracy: {clean_pf:.3f}%")
        print(f"Candidates in first {L} layers: {nC_pf}")
        print("\nAR(k) (cumulative flips):")
        print("k\tAcc_k (%)\tAR(k)")
        
        for k in KS:
            print(f"{k}\t{acck_pf[k]:.3f}\t\t{ark_pf[k]*100:.2f}%")

# Expirment #3 - pass free attack on Q layers only.
## we only aplyed it to the Query layers 
we tryed difrent values of K.

In [32]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (1, 3, 6, 12)
run_experiment(KS, LS, exp_num = 3, exp_collector = "passfree", exp_stratagy= "Q")

--- Run Expiriment #3 for L = 1 ---
---LOGGING LAYERS----
Layers actually collected: 1
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0012406947890819732, 1000: 0.0024813895781637925, 10000: 0.004962779156327585, 100000: 0.006203473945409559}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 1 layers: 589824

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.317		0.12%
1000	92.202		0.25%
10000	91.972		0.50%
100000	91.858		0.62%
--- Run Expiriment #3 for L = 3 ---
---LOGGING LAYERS----
Layers actually collected: 3
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.003722084367245766, 10000: 0.0074441687344913784, 100000: 0.03722084367245658}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 3 layers: 1769472

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.4

# Expirment #4 - pass free attack on K layers only.
## we only aplyed it to the Key layers 
we tryed difrent values of K.

In [11]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (1, 3, 6, 12)
run_experiment(KS, LS, exp_num = 4, exp_collector = "passfree", exp_stratagy= "K")

--- Run Expiriment #4 for L = 1 ---
---LOGGING LAYERS----
Layers actually collected: 1
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0, 1000: -0.0012406947890818195, 10000: -0.002481389578163639, 100000: 0.0024813895781637925}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 1 layers: 589824

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.546		-0.12%
10000	92.661		-0.25%
100000	92.202		0.25%
--- Run Expiriment #4 for L = 3 ---
---LOGGING LAYERS----
Layers actually collected: 3
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.0, 10000: 0.003722084367245766, 100000: 0.03101736972704718}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 3 layers: 1769472

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.431		0.00%
10

# Expirment #5 - pass free attack on V layers only.
## we only aplyed it to the Value layers 
we tryed difrent values of K.

In [12]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (1, 3, 6, 12)
run_experiment(KS, LS, exp_num = 5, exp_collector = "passfree", exp_stratagy= "V")

--- Run Expiriment #5 for L = 1 ---
---LOGGING LAYERS----
Layers actually collected: 1
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.0, 10000: 0.003722084367245766, 100000: 0.01116625310173699}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 1 layers: 589824

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.431		0.00%
10000	92.087		0.37%
100000	91.399		1.12%
--- Run Expiriment #5 for L = 3 ---
---LOGGING LAYERS----
Layers actually collected: 3
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0012406947890819732, 1000: -0.0012406947890818195, 10000: -0.0012406947890818195, 100000: 0.026054590570719745}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 3 layers: 1769472

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.317		0.12%
1000	

# Expirment #6 - pass free attack on QK layers.
## we  aplyed it to the Q and K layers 
we tryed difrent values of K.

In [13]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (2, 4, 8)
run_experiment(KS, LS, exp_num = 6, exp_collector = "passfree", exp_stratagy= "QK")

--- Run Expiriment #6 for L = 2 ---
---LOGGING LAYERS----
Layers actually collected: 2
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.0024813895781637925, 10000: 0.003722084367245766, 100000: -0.0012406947890818195}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 2 layers: 1179648

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.202		0.25%
10000	92.087		0.37%
100000	92.546		-0.12%
--- Run Expiriment #6 for L = 4 ---
---LOGGING LAYERS----
Layers actually collected: 4
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.003722084367245766, 10000: 0.003722084367245766, 100000: 0.0074441687344913784}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 4 layers: 2359296

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
100

# Expirment #7 - pass free attack on QV layers.
## we aplyed it to the Q and v layers
we tryed difrent values of K.

In [12]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (2, 4, 8)
run_experiment(KS, LS, exp_num = 7, exp_collector = "passfree", exp_stratagy= "QV")

--- Run Expiriment #7 for L = 2 ---
---LOGGING LAYERS----
Layers actually collected: 2
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0012406947890819732, 1000: 0.0012406947890819732, 10000: 0.0012406947890819732, 100000: 0.006203473945409559}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 2 layers: 1179648

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.317		0.12%
1000	92.317		0.12%
10000	92.317		0.12%
100000	91.858		0.62%
--- Run Expiriment #7 for L = 4 ---
---LOGGING LAYERS----
Layers actually collected: 4
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0012406947890819732, 1000: 0.0012406947890819732, 10000: 0.008684863523573198, 100000: 0.013647642679900783}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 4 layers: 2359296

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	9

# Expirment #8 - pass free attack on KV layers.
## we aplyed it to the K and V layers
we tryed difrent values of K.

In [11]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (2, 4, 8)
run_experiment(KS, LS, exp_num = 8, exp_collector = "passfree", exp_stratagy= "KV")

--- Run Expiriment #8 for L = 2 ---
---LOGGING LAYERS----
Layers actually collected: 2
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0, 1000: -0.0012406947890818195, 10000: 0.0, 100000: 0.0024813895781637925}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 2 layers: 1179648

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.546		-0.12%
10000	92.431		0.00%
100000	92.202		0.25%
--- Run Expiriment #8 for L = 4 ---
---LOGGING LAYERS----
Layers actually collected: 4
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.006203473945409559, 10000: 0.0074441687344913784, 100000: 0.003722084367245766}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 4 layers: 2359296

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	91.858		0.62%
1

# Expirment #9 - pass free attack on KQV layers.
## we aplyed it to the Q, K and V layers
we tryed difrent values of K.

In [12]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (3, 6, 9)
run_experiment(KS, LS, exp_num = 9, exp_collector = "passfree", exp_stratagy= "KQV")

--- Run Expiriment #9 for L = 3 ---
---LOGGING LAYERS----
Layers actually collected: 3
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.0024813895781637925, 10000: 0.003722084367245766, 100000: -0.003722084367245612}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 3 layers: 1769472

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.202		0.25%
10000	92.087		0.37%
100000	92.775		-0.37%
--- Run Expiriment #9 for L = 6 ---
---LOGGING LAYERS----
Layers actually collected: 6
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.003722084367245766, 10000: 0.00992555831265517, 100000: 0.008684863523573198}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 6 layers: 3538944

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	9

# Expirment #10 - pass free attack on FFN layers.
## we aplyed it to the FFN layers
we tryed difrent values of K.

In [15]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (2, 4, 8)
run_experiment(KS, LS, exp_num = 10, exp_collector = "passfree", exp_stratagy= "FFN")

--- Run Expiriment #10 for L = 2 ---
---LOGGING LAYERS----
Layers actually collected: 0
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.0, 10000: 0.0, 100000: 0.0}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 2 layers: 0

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.431		0.00%
10000	92.431		0.00%
100000	92.431		0.00%
--- Run Expiriment #10 for L = 4 ---
---LOGGING LAYERS----
Layers actually collected: 0
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0, 100: 0.0, 1000: 0.0, 10000: 0.0, 100000: 0.0}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 4 layers: 0

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.431		0.00%
10000	92.431		0.00%
100000	92.431		0.00%
--- Run Expiriment #10 for L = 8 ---
---LOGGING LAYERS----
L

# Expirment #11 - pass free attack on the classifier layer.
## we aplyed it to the classifier layer
we tryed difrent values of K.

In [11]:
KS = (1,10,100,500,1_000, 2_000)
LS = (1,)
run_experiment(KS, LS, exp_num = 11, exp_collector = "passfree", exp_stratagy= "classifier")

--- Run Expiriment #11 for L = 1 ---
---LOGGING LAYERS----
Layers actually collected: 1
---END LOGGING LAYERS----
{1: 0.0, 10: 0.0024813895781637925, 100: 0.0074441687344913784, 500: 0.9168734491315136, 1000: 0.9168734491315136, 2000: 0.9181141439205956}
Evaluating passfree attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 1 layers: 1536

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.202		0.25%
100	91.743		0.74%
500	7.683		91.69%
1000	7.683		91.69%
2000	7.569		91.81%


# from hear its the one-pass expirinets

In [11]:

def run_expiriment_one_pass(KS_list, LS_list, exp_num, exp_stratagy):
    for L in LS_list:
        print(f"--- Run Expiriment #{exp_num} for L = {L} ---")
        
        model_llm = load_bert_sst2_llm()
    
        dnl_1pass_prepare_grads_llm(model_llm, tokenizer_llm, device=DEVICE, batch_size=16, seq_len=128)
        # (Optional) quick debug: restrict eval set size earlier with max_eval=2000 in load_sst2_llm
        clean_pf, acck_pf, ark_pf, nC_pf = eval_dnl_ARk_llm(model_llm, eval_loader_llm, L=L, ks=KS, collector='1pass', stratagy = exp_stratagy)
        
        print(f"Evaluating 1pass attack on BERT (SST-2):")
        print(f"Evaluated on {len(eval_loader_llm.dataset)} examples.")
        print(f"Baseline top-1 accuracy: {clean_pf:.3f}%")
        print(f"Candidates in first {L} layers: {nC_pf}")
        print("\nAR(k) (cumulative flips):")
        print("k\tAcc_k (%)\tAR(k)")
        for k in KS_list:
            print(f"{k}\t{acck_pf[k]:.3f}\t\t{ark_pf[k]*100:.2f}%")

# Expirment #12 - 1pass attack on the K layer.
we tryed difrent values of K.

In [14]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (1, 3, 6, 12)
run_expiriment_one_pass(KS, LS, exp_num = 12, exp_stratagy= "K")

--- Run Expiriment #12 for L = 1 ---
---LOGGING LAYERS----
Layers actually collected: 1
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 1 layers: 589824

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.546		-0.12%
10000	92.546		-0.12%
100000	92.317		0.12%
--- Run Expiriment #12 for L = 3 ---
---LOGGING LAYERS----
Layers actually collected: 3
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 3 layers: 1769472

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.317		0.12%
10000	92.087		0.37%
100000	89.794		2.85%
--- Run Expiriment #12 for L = 6 ---
---LOGGING LAYERS----
Layers actually collected: 6
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.

# Expirment #13 - 1pass attack on the Q layer.
we tryed difrent values of Q.

In [11]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (1, 3, 6, 12)
run_expiriment_one_pass(KS, LS, exp_num = 13, exp_stratagy= "Q")

--- Run Expiriment #13 for L = 1 ---
---LOGGING LAYERS----
Layers actually collected: 1
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 1 layers: 589824

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.317		0.12%
1000	92.317		0.12%
10000	92.087		0.37%
100000	91.743		0.74%
--- Run Expiriment #13 for L = 3 ---
---LOGGING LAYERS----
Layers actually collected: 3
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 3 layers: 1769472

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.317		0.12%
10000	91.743		0.74%
100000	88.991		3.72%
--- Run Expiriment #13 for L = 6 ---
---LOGGING LAYERS----
Layers actually collected: 6
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
B

# Expirment #14 - 1pass attack on the V layer.
we tryed difrent values of V.

In [12]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (1, 3, 6, 12)
run_expiriment_one_pass(KS, LS, exp_num = 14, exp_stratagy= "V")

--- Run Expiriment #14 for L = 1 ---
---LOGGING LAYERS----
Layers actually collected: 1
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 1 layers: 589824

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.431		0.00%
10000	92.087		0.37%
100000	91.628		0.87%
--- Run Expiriment #14 for L = 3 ---
---LOGGING LAYERS----
Layers actually collected: 3
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 3 layers: 1769472

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.317		0.12%
10	92.317		0.12%
100	92.661		-0.25%
1000	92.317		0.12%
10000	92.775		-0.37%
100000	89.106		3.60%
--- Run Expiriment #14 for L = 6 ---
---LOGGING LAYERS----
Layers actually collected: 6
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.

# Expirment #15 - 1pass attack on the K,Q layer.
we tryed difrent values of K.

In [13]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (2, 4, 8)
run_expiriment_one_pass(KS, LS, exp_num = 15, exp_stratagy= "KQ")

--- Run Expiriment #15 for L = 2 ---
---LOGGING LAYERS----
Layers actually collected: 2
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 2 layers: 1179648

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.202		0.25%
10000	92.202		0.25%
100000	92.317		0.12%
--- Run Expiriment #15 for L = 4 ---
---LOGGING LAYERS----
Layers actually collected: 4
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 4 layers: 2359296

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.087		0.37%
10000	91.858		0.62%
100000	91.858		0.62%
--- Run Expiriment #15 for L = 8 ---
---LOGGING LAYERS----
Layers actually collected: 8
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.


# Expirment #16 - 1pass attack on the KV layer.
we tryed difrent values of K.

In [11]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (2,4,8)
run_expiriment_one_pass(KS, LS, exp_num = 16, exp_stratagy= "KV")

--- Run Expiriment #16 for L = 2 ---
---LOGGING LAYERS----
Layers actually collected: 2
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 2 layers: 1179648

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.546		-0.12%
10000	92.546		-0.12%
100000	92.431		0.00%
--- Run Expiriment #16 for L = 4 ---
---LOGGING LAYERS----
Layers actually collected: 4
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 4 layers: 2359296

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.202		0.25%
1000	91.858		0.62%
10000	91.858		0.62%
100000	92.202		0.25%
--- Run Expiriment #16 for L = 8 ---
---LOGGING LAYERS----
Layers actually collected: 8
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples

# Experiment #17 - 1pass attack on the QV layer.
we tryed difrent values of K.

In [12]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (2,4,8)
run_expiriment_one_pass(KS, LS, exp_num = 17, exp_stratagy= "QV")

--- Run Expiriment #17 for L = 2 ---
---LOGGING LAYERS----
Layers actually collected: 2
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 2 layers: 1179648

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.317		0.12%
1000	92.317		0.12%
10000	92.202		0.25%
100000	91.858		0.62%
--- Run Expiriment #17 for L = 4 ---
---LOGGING LAYERS----
Layers actually collected: 4
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 4 layers: 2359296

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.317		0.12%
1000	92.431		0.00%
10000	91.858		0.62%
100000	91.284		1.24%
--- Run Expiriment #17 for L = 8 ---
---LOGGING LAYERS----
Layers actually collected: 8
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.


# Experiment #18 - 1pass attack on the KQV layer.
we tryed difrent values of K.

In [13]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (3, 6, 9)
run_expiriment_one_pass(KS, LS, exp_num = 18, exp_stratagy= "KQV")

--- Run Expiriment #18 for L = 3 ---
---LOGGING LAYERS----
Layers actually collected: 3
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 3 layers: 1769472

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.202		0.25%
10000	91.972		0.50%
100000	92.661		-0.25%
--- Run Expiriment #18 for L = 6 ---
---LOGGING LAYERS----
Layers actually collected: 6
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 6 layers: 3538944

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.087		0.37%
10000	91.284		1.24%
100000	91.858		0.62%
--- Run Expiriment #18 for L = 9 ---
---LOGGING LAYERS----
Layers actually collected: 9
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.

# Experiment #19 - 1pass attack on the FFN layer.
we tryed difrent values of K.

In [14]:
KS = (1,10,100,1_000,10_000, 100_000)
LS = (2,4,8)
run_expiriment_one_pass(KS, LS, exp_num = 19, exp_stratagy= "FFN")

--- Run Expiriment #19 for L = 2 ---
---LOGGING LAYERS----
Layers actually collected: 0
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 2 layers: 0

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.431		0.00%
10000	92.431		0.00%
100000	92.431		0.00%
--- Run Expiriment #19 for L = 4 ---
---LOGGING LAYERS----
Layers actually collected: 0
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 4 layers: 0

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.431		0.00%
100	92.431		0.00%
1000	92.431		0.00%
10000	92.431		0.00%
100000	92.431		0.00%
--- Run Expiriment #19 for L = 8 ---
---LOGGING LAYERS----
Layers actually collected: 0
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top

# Experiment #20 - 1pass attack on the classifier layer.
we tryed difrent values of K.

In [15]:
KS = (1,10,100,500,1_000, 2_000)
LS = (1,)
run_expiriment_one_pass(KS, LS, exp_num = 20 , exp_stratagy= "classifier")

--- Run Expiriment #20 for L = 1 ---
---LOGGING LAYERS----
Layers actually collected: 1
---END LOGGING LAYERS----
Evaluating 1pass attack on BERT (SST-2):
Evaluated on 872 examples.
Baseline top-1 accuracy: 92.431%
Candidates in first 1 layers: 1536

AR(k) (cumulative flips):
k	Acc_k (%)	AR(k)
1	92.431		0.00%
10	92.202		0.25%
100	92.317		0.12%
500	7.798		91.56%
1000	7.913		91.44%
2000	7.569		91.81%


# Old and not relevante!

In [23]:
import re
import torch
import torch.nn as nn
from typing import List, Tuple, Dict

def _role_from_name_llm(name: str, module: nn.Module) -> str:
    """
    Classify BERT module roles by dotted path:
      Q/K/V, AttnOut, FFN1 (intermediate), FFN2 (output), CLS (classifier), EMB (embeddings), LN (layernorm)
    """
    if isinstance(module, nn.Linear):
        if ".attention.self.query" in name:   return "Q"
        elif ".attention.self.key"   in name:   return "K"
        elif ".attention.self.value" in name:   return "V"
        elif ".attention.output.dense" in name: return "AttnOut"
        # FFN parts (exclude attention output)
        elif ".intermediate.dense" in name:               return "FFN1"
        elif ".output.dense" in name and ".attention." not in name: return "FFN2"
        elif name.endswith("classifier"):       return "CLS"
    if isinstance(module, nn.Embedding):
        return "EMB"
    if isinstance(module, nn.LayerNorm):
        return "LN"
    return "OTHER"

def _iter_param_modules_llm(model: nn.Module) -> List[Tuple[str, nn.Module, str]]:
    """Yield (name, module, role) for parameter-bearing modules we care about."""
    for name, m in model.named_modules():
        if isinstance(m, (nn.Linear, nn.Embedding)):
            if getattr(m, "weight", None) is not None:
                yield name, m, _role_from_name_llm(name, m)

def _select_modules_by_strategy_llm(model: nn.Module, strategy: str, L: int) -> List[nn.Module]:
    """
    Filter parameter modules by strategy, preserve forward order, then keep first L.
      strategies:
        - 'attn_Q_only' / 'attn_K_only' / 'attn_V_only'
        - 'attn_QKV_interlace' (handled specially in collector; here we pass Q+K+V)
        - 'mlp_only'  (FFN1 + FFN2)
        - 'classifier_head_only' (CLS)
    """
    roles_keep = set()
    if strategy == "attn_Q_only": roles_keep = {"Q"}
    elif strategy == "attn_K_only": roles_keep = {"K"}
    elif strategy == "attn_V_only": roles_keep = {"V"}
    elif strategy == "attn_QKV_interlace": roles_keep = {"Q","K","V"}
    elif strategy == "mlp_only": roles_keep = {"FFN1","FFN2"}
    elif strategy == "classifier_head_only": roles_keep = {"CLS"}
    else:
        raise ValueError(f"Unknown strategy '{strategy}'")

    kept = []
    for name, m, role in _iter_param_modules_llm(model):
        if role in roles_keep:
            kept.append(m)
    return kept[:L]


In [24]:
def _rowwise_argmax_candidates_llm(score_tensor: torch.Tensor, module: nn.Module):
    """
    From score tensor shaped like weight (rows x cols), select 1 element per row with max score.
    Returns list[(score, (row, col), module)].
    """
    vmax, arg = score_tensor.max(dim=1)
    cands = []
    rows = score_tensor.size(0)
    for r in range(rows):
        s = float(vmax[r])
        if s == 0.0:
            continue
        j = int(arg[r])
        cands.append((s, (r, j), module))
    return cands

def collect_passfree_candidates_llm_filtered(modules: List[nn.Module]):
    """Pass-free: |w| per row (one-per-kernel)."""
    out = []
    for m in modules:
        W = m.weight.data
        A = W.abs()
        out.extend(_rowwise_argmax_candidates_llm(A, m))
    return out

def collect_1pass_candidates_llm_filtered(modules: List[nn.Module]):
    """1-pass: S = |w| + w*g + 0.5*w^2*g^2 per row (one-per-kernel)."""
    out = []
    for m in modules:
        W = m.weight.data
        G = getattr(m.weight, "grad", None)
        if G is None:
            raise RuntimeError("collect_1pass_candidates_llm_filtered: grads missing. Call dnl_1pass_prepare_grads_llm first.")
        S = W.abs() + (W * G) + 0.5 * (W * W) * (G * G)
        out.extend(_rowwise_argmax_candidates_llm(S, m))
    return out


In [25]:
def collect_interlace_QKV_candidates_llm(model: nn.Module, L: int, mode: str = "pass_free"):
    """
    Build three candidate lists (Q, K, V), sort each by score desc, then round-robin merge.
    mode: 'pass_free' or '1pass'
    """
    # pick modules for each role, honoring first-L within each role in forward order
    mods_Q, mods_K, mods_V = [], [], []
    for name, m, role in _iter_param_modules_llm(model):
        if role == "Q": mods_Q.append(m)
        elif role == "K": mods_K.append(m)
        elif role == "V": mods_V.append(m)
    mods_Q, mods_K, mods_V = mods_Q[:L], mods_K[:L], mods_V[:L]

    if mode == "pass_free":
        build = collect_passfree_candidates_llm_filtered
    elif mode == "1pass":
        build = collect_1pass_candidates_llm_filtered
    else:
        raise ValueError("mode must be 'pass_free' or '1pass'")

    cq = sorted(build(mods_Q), key=lambda t: t[0], reverse=True)
    ck = sorted(build(mods_K), key=lambda t: t[0], reverse=True)
    cv = sorted(build(mods_V), key=lambda t: t[0], reverse=True)

    # round-robin merge
    merged, i, j, k = [], 0, 0, 0
    while i < len(cq) or j < len(ck) or k < len(cv):
        if i < len(cq): 
            merged.append(cq[i]); i += 1
        if j < len(ck): 
            merged.append(ck[j]); j += 1
        if k < len(cv): 
            merged.append(cv[k]); k += 1
    return merged


In [26]:
def eval_dnl_ARk_llm_strategy(model,
                              loader,
                              tokenizer,
                              L:int=10,
                              ks:Tuple[int,...]=(1,2,3,4,5),
                              device=DEVICE,
                              mode:str="pass_free",
                              strategy:str="attn_Q_only"):
    """
    mode: 'pass_free' or '1pass'
    strategy: one of
      'attn_Q_only','attn_K_only','attn_V_only',
      'attn_QKV_interlace','mlp_only','classifier_head_only'
    """
    model.to(device).eval()

    # 1) clean accuracy
    clean = accuracy_llm(model, loader, device)

    # 2) prepare grads if 1-pass
    if mode == "1pass":
        dnl_1pass_prepare_grads_llm(model, tokenizer, device=device, batch_size=16, seq_len=128)

    # 3) build candidate list according to strategy
    if strategy == "attn_QKV_interlace":
        cands = collect_interlace_QKV_candidates_llm(model, L=L, mode=("1pass" if mode=="1pass" else "pass_free"))
    else:
        modules = _select_modules_by_strategy_llm(model, strategy=strategy, L=L)
        if mode == "pass_free":
            cands = collect_passfree_candidates_llm_filtered(modules)
        else:
            cands = collect_1pass_candidates_llm_filtered(modules)

        cands.sort(key=lambda t: t[0], reverse=True)

    # 4) cumulative flips + eval
    acc_k = {}
    flipped = 0
    for k in ks:
        while flipped < k and flipped < len(cands):
            _, idx, mod = cands[flipped]
            flip_sign_llm(mod, idx)
            flipped += 1
        acc_k[k] = accuracy_llm(model, loader, device)

    ar_k = {k: (clean - acc_k[k]) / max(1e-12, clean) for k in ks}
    return clean, acc_k, ar_k, len(cands)


In [27]:
def print_results_strategy_llm(attack_name:str,
                               clean:float,
                               acc_k:Dict[int,float],
                               ar_k:Dict[int,float],
                               num_cands:int,
                               L:int,
                               KS:Tuple[int,...],
                               eval_loader):
    print(f"Evaluating {attack_name} attack on BERT:")
    print(f"Evaluated on {len(eval_loader.dataset)} samples.")
    print(f"Baseline top-1 accuracy: {clean:.3f}%")
    print(f"Candidates in first {L} layers: {num_cands}")
    print("\nAR(k) (cumulative flips):")
    print("k\tAcc_k (%)\tAR(k)")
    for k in KS:
        print(f"{k}\t{acc_k[k]:.3f}\t\t{ar_k[k]*100:.2f}%")


In [29]:
for name, m, role in _iter_param_modules_llm(model_llm):
    print(f"Cur layer module is: {name} m = {m} role = {role}")


Cur layer module is: bert.embeddings.word_embeddings m = Embedding(30522, 768, padding_idx=0) role = EMB
Cur layer module is: bert.embeddings.position_embeddings m = Embedding(512, 768) role = EMB
Cur layer module is: bert.embeddings.token_type_embeddings m = Embedding(2, 768) role = EMB
Cur layer module is: bert.encoder.layer.0.attention.self.query m = Linear(in_features=768, out_features=768, bias=True) role = Q
Cur layer module is: bert.encoder.layer.0.attention.self.key m = Linear(in_features=768, out_features=768, bias=True) role = K
Cur layer module is: bert.encoder.layer.0.attention.self.value m = Linear(in_features=768, out_features=768, bias=True) role = V
Cur layer module is: bert.encoder.layer.0.attention.output.dense m = Linear(in_features=768, out_features=768, bias=True) role = AttnOut
Cur layer module is: bert.encoder.layer.0.intermediate.dense m = Linear(in_features=768, out_features=3072, bias=True) role = FFN1
Cur layer module is: bert.encoder.layer.0.output.dense m =